In [32]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [33]:
all_data = np.loadtxt('diabetes.csv.gz',delimiter=',',dtype=np.float32)
all_data

array([[-0.294118 ,  0.487437 ,  0.180328 , ..., -0.53117  , -0.0333333,
         0.       ],
       [-0.882353 , -0.145729 ,  0.0819672, ..., -0.766866 , -0.666667 ,
         1.       ],
       [-0.0588235,  0.839196 ,  0.0491803, ..., -0.492741 , -0.633333 ,
         0.       ],
       ...,
       [-0.411765 ,  0.21608  ,  0.180328 , ..., -0.857387 , -0.7      ,
         1.       ],
       [-0.882353 ,  0.266332 , -0.0163934, ..., -0.768574 , -0.133333 ,
         0.       ],
       [-0.882353 , -0.0653266,  0.147541 , ..., -0.797609 , -0.933333 ,
         1.       ]], shape=(759, 9), dtype=float32)

In [34]:
x_data = all_data[:,:-1]
y_data = all_data[:,[-1]]

train_x = torch.tensor(x_data,dtype=torch.float32,device='cuda')
train_y = torch.tensor(y_data,dtype=torch.float32,device='cuda')

In [35]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.linear1 = torch.nn.Linear(8, 6, bias=True)
        self.linear2 = torch.nn.Linear(6, 2, bias=True)
        self.linear3 = torch.nn.Linear(2, 1, bias=True)
        self.hidden_act = torch.nn.ReLU()
        self.output_act = torch.nn.Sigmoid()

    def forward(self, x):
        x = self.hidden_act(self.linear1(x))
        x = self.hidden_act(self.linear2(x))
        return self.output_act(self.linear3(x))


model = NeuralNetwork().to('cuda')
criterion = torch.nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.01,weight_decay=1e-4)

In [36]:
prev_params = {name: p.detach().clone() for name, p in model.named_parameters()}
for epoch in range(1,10001):
    y_hat = model(train_x)
    loss = criterion(y_hat,train_y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        delta_norms = {}
        for name, p in model.named_parameters():
            delta = p - prev_params[name]
            delta_norms[name] = delta.norm().item()
            prev_params[name] = p.detach().clone()

    delta_msg = ", ".join([f"{k}: {v:.6f}" for k, v in delta_norms.items()])
    print(f"epoch:{epoch}, loss:{loss.item():.6f}, {delta_msg}")

epoch:1, loss:0.858365, linear1.weight: 0.069274, linear1.bias: 0.024495, linear2.weight: 0.034622, linear2.bias: 0.014140, linear3.weight: 0.014141, linear3.bias: 0.010000
epoch:2, loss:0.853657, linear1.weight: 0.067242, linear1.bias: 0.024035, linear2.weight: 0.034101, linear2.bias: 0.014049, linear3.weight: 0.014008, linear3.bias: 0.009997
epoch:3, loss:0.849404, linear1.weight: 0.062824, linear1.bias: 0.023087, linear2.weight: 0.032992, linear2.bias: 0.013795, linear3.weight: 0.013739, linear3.bias: 0.009993
epoch:4, loss:0.845602, linear1.weight: 0.057922, linear1.bias: 0.021506, linear2.weight: 0.031255, linear2.bias: 0.013327, linear3.weight: 0.013343, linear3.bias: 0.009987
epoch:5, loss:0.842157, linear1.weight: 0.052233, linear1.bias: 0.019482, linear2.weight: 0.029338, linear2.bias: 0.012735, linear3.weight: 0.012881, linear3.bias: 0.009980
epoch:6, loss:0.838932, linear1.weight: 0.046318, linear1.bias: 0.017312, linear2.weight: 0.027677, linear2.bias: 0.012149, linear3.wei

In [37]:
model.eval()
with torch.no_grad():
    y_prob = model(train_x)
    loss_eval = criterion(y_prob, train_y).item()
    y_pred = (y_prob >= 0.5).float()                                                                                                    
    acc = (y_pred == train_y).float().mean().item()

print(f"Eval loss: {loss_eval:.6f}")
print(f"Eval acc:  {acc:.4f}")

Eval loss: 0.393240
Eval acc:  0.8195
